# 摩洛哥智能电表高分辨率用电负荷数据预处理与分析

**规范文档**: `docs/摩洛哥多城市电力负荷全流程分析流程设计.md`
**执行Prompt**: `docs/Morocco_Load_Analysis_Execution_Prompts.md`

本Notebook对应 **Prompt-01: 数据预处理**，覆盖9个处理步骤。

## 0. 全局配置与路径设置

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
from datetime import datetime

# === 中文字体设置 ===
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 12

# === 路径配置 ===
DATA_DIR = 'data/'
RAW_DATA_FILE = os.path.join(DATA_DIR, 'Data Morocco.xlsx')
OUTPUT_DIR = 'outputs/ch01_data_preprocessing/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === 城市配置 ===
CITIES = {
    'Laayoune':    {'sheet': 'Laayoune',    'sampling': '10min', 'unit': 'A',  'zones': 5},
    'Boujdour':    {'sheet': 'Boujdour',    'sampling': '10min', 'unit': 'A',  'zones': 3},
    'Foum eloued': {'sheet': 'Foum eloued', 'sampling': '10min', 'unit': 'A',  'zones': 7},
    'Marrakech':   {'sheet': 'Marrakech',   'sampling': '30min', 'unit': 'kW', 'zones': 2},
}

# === 量纲转换参数 ===
VOLTAGE = 220          # 市电标准电压 V
POWER_FACTOR = 0.9     # 平均功率因数

# === 季节映射（北半球） ===
SEASON_MAP = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring',  4: 'Spring',  5: 'Spring',
    6: 'Summer',  7: 'Summer',  8: 'Summer',
    9: 'Autumn',  10: 'Autumn', 11: 'Autumn'
}

print('全局配置加载完成')
print(f'原始数据路径: {RAW_DATA_FILE}')
print(f'输出目录: {OUTPUT_DIR}')

## Step 1.1: 数据读取与结构探查

**目标**: 批量读取4个Sheet，检查行数、列数、数据类型、缺失值、数值范围
**产物**: `ch01_data_profile_report.md`

In [ ]:
# 批量读取4个城市的原始数据
city_data = {}
profile_lines = ['# 数据概况报告\n\n']
profile_lines.append(f'生成时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')

for city_name, config in CITIES.items():
    df = pd.read_excel(RAW_DATA_FILE, sheet_name=config['sheet'], engine='openpyxl')
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    df = df.set_index('DateTime').sort_index()
    df['city'] = city_name
    city_data[city_name] = df
    
    # 记录概况
    profile_lines.append(f'## {city_name}\n')
    profile_lines.append(f'- Sheet名: {config["sheet"]}\n')
    profile_lines.append(f'- 采样间隔: {config["sampling"]}\n')
    profile_lines.append(f'- 原始单位: {config["unit"]}\n')
    profile_lines.append(f'- 数据量: {len(df)} 行 × {len(df.columns)} 列\n')
    profile_lines.append(f'- 时间范围: {df.index.min()} ~ {df.index.max()}\n')
    profile_lines.append(f'- Zone列: {df.drop(columns=["city"]).columns.tolist()}\n')
    profile_lines.append(f'- 缺失值: {df.drop(columns=["city"]).isnull().sum().sum()} 个\n')
    profile_lines.append(f'- 数据类型: {df.drop(columns=["city"]).dtypes.unique().tolist()}\n\n')
    
    print(f'✓ {city_name}: {len(df)} 行, {len(df.columns)-1} zones, {config["sampling"]}, {config["unit"]}')

# 保存数据概况报告
profile_report_path = os.path.join(OUTPUT_DIR, 'ch01_data_profile_report.md')
with open(profile_report_path, 'w', encoding='utf-8') as f:
    f.writelines(profile_lines)
print(f'\n数据概况报告已保存: {profile_report_path}')

## Step 1.2: 缺失值检测与统计

**目标**: 计算每列缺失率，缺失率>5%需特别报告
**产物**: `ch01_missing_stats.csv`

In [ ]:
missing_records = []

for city_name, df in city_data.items():
    zone_cols = [c for c in df.columns if c != 'city']
    for col in zone_cols:
        missing_count = df[col].isnull().sum()
        missing_rate = missing_count / len(df) * 100
        missing_records.append({
            'city': city_name,
            'zone': col,
            'missing_count': missing_count,
            'total_rows': len(df),
            'missing_rate_pct': round(missing_rate, 4)
        })
        if missing_rate > 5:
            print(f'⚠ {city_name}/{col}: 缺失率 {missing_rate:.2f}% > 5%，需特别关注！')

missing_df = pd.DataFrame(missing_records)
missing_path = os.path.join(OUTPUT_DIR, 'ch01_missing_stats.csv')
missing_df.to_csv(missing_path, index=False)
print(f'\n缺失值统计表已保存: {missing_path}')
print(f'总缺失值: {missing_df["missing_count"].sum()}')

## Step 1.3: 时间戳解析与标准化

**目标**: 统一为datetime64类型，设置为索引，按时间升序排列
**产物**: 无独立文件（内嵌于后续步骤）

In [ ]:
# 时间戳已在Step 1.1中完成解析和标准化
# 此处验证时间戳质量
for city_name, df in city_data.items():
    time_diffs = pd.Series(df.index).diff().dropna()
    most_common = time_diffs.value_counts().head(3)
    print(f'{city_name}:')
    print(f'  时间范围: {df.index.min()} ~ {df.index.max()}')
    print(f'  最常见时间间隔: {most_common}')
    print()

## Step 1.4: 时序对齐（Marrakech 30min → 10min 上采样）

**目标**: 将Marrakech从30分钟线性插值上采样至10分钟，与其他三城市对齐
**方法**: `resample('10T').interpolate(method='linear')`
**产物**: `ch01_marrakech_resampled.csv`

In [ ]:
print(f'Marrakech 上采样前: {len(city_data["Marrakech"])} 行')

# Marrakech 30min → 10min 线性插值上采样
marrakech_df = city_data['Marrakech'].copy()
marrakech_city_col = marrakech_df['city']
marrakech_zones = marrakech_df.drop(columns=['city'])

marrakech_resampled = marrakech_zones.resample('10T').interpolate(method='linear', limit_direction='both')
marrakech_resampled['city'] = 'Marrakech'
city_data['Marrakech'] = marrakech_resampled

print(f'Marrakech 上采样后: {len(city_data["Marrakech"])} 行')

# 保存上采样数据
resampled_path = os.path.join(OUTPUT_DIR, 'ch01_marrakech_resampled.csv')
city_data['Marrakech'].to_csv(resampled_path)
print(f'Marrakech上采样数据已保存: {resampled_path}')

## Step 1.5: 量纲统一（A → kW）

**目标**: 将三城市电流数据(A)换算为有功功率(kW)，与Marrakech统一
**公式**: P(kW) = I(A) × 220(V) × 0.9 / 1000
**产物**: `ch01_unified_power_all_cities.csv`

In [ ]:
for city_name, config in CITIES.items():
    if config['unit'] == 'A':
        zone_cols = [c for c in city_data[city_name].columns if c != 'city']
        # P(kW) = I(A) × V(V) × cos(φ) / 1000
        city_data[city_name][zone_cols] = (
            city_data[city_name][zone_cols] * VOLTAGE * POWER_FACTOR / 1000
        )
        print(f'✓ {city_name}: 电流(A) → 有功功率(kW) 换算完成')
        print(f'  转换后数值范围示例 ({zone_cols[0]}): '
              f'{city_data[city_name][zone_cols[0]].min():.2f} ~ {city_data[city_name][zone_cols[0]].max():.2f} kW')
    else:
        print(f'✓ {city_name}: 已是kW单位，无需换算')

# 合并所有城市数据
all_cities_df = pd.concat(city_data.values(), axis=0)
all_cities_df = all_cities_df.sort_index()

# 保存量纲统一后的全城市数据
unified_path = os.path.join(OUTPUT_DIR, 'ch01_unified_power_all_cities.csv')
all_cities_df.to_csv(unified_path)
print(f'\n量纲统一全城市数据已保存: {unified_path}')
print(f'合并后总数据量: {len(all_cities_df)} 行')

## Step 1.6: 异常值检测（3σ准则）

**目标**: 按城市+zone分别计算3σ阈值，标记异常点（不直接删除）
**方法**: 超出 [μ-3σ, μ+3σ] 范围的数据标记为异常
**产物**: `ch01_outlier_flags.csv`

In [ ]:
outlier_flags = all_cities_df[['city']].copy()
zone_cols = [c for c in all_cities_df.columns if c != 'city']

for city_name in CITIES.keys():
    city_mask = all_cities_df['city'] == city_name
    city_subset = all_cities_df.loc[city_mask, zone_cols]
    
    for col in zone_cols:
        col_data = city_subset[col]
        mean_val = col_data.mean()
        std_val = col_data.std()
        lower = mean_val - 3 * std_val
        upper = mean_val + 3 * std_val
        
        flag_col = f'{col}_outlier'
        outlier_flags[flag_col] = False
        outlier_flags.loc[city_mask, flag_col] = (
            (col_data < lower) | (col_data > upper)
        ).values
        
        outlier_count = outlier_flags.loc[city_mask, flag_col].sum()
        outlier_rate = outlier_count / city_mask.sum() * 100
        print(f'{city_name}/{col}: {outlier_count} 个异常点 ({outlier_rate:.2f}%)')

# 保存异常值标记表
outlier_path = os.path.join(OUTPUT_DIR, 'ch01_outlier_flags.csv')
outlier_flags.to_csv(outlier_path)
print(f'\n异常值标记表已保存: {outlier_path}')

## Step 1.7: 异常值处理（线性插值替换）+ 负值处理

**目标**: 将被标记的异常点用线性插值替换，确保无NaN残留
**方法**: 先将异常值设为NaN，再 `interpolate(method='linear')` 填充
**产物**: `ch01_cleaned_data.csv`

In [ ]:
df_cleaned = all_cities_df.copy()

# 将异常值设为NaN
outlier_flag_cols = [c for c in outlier_flags.columns if c.endswith('_outlier')]
for flag_col in outlier_flag_cols:
    zone_col = flag_col.replace('_outlier', '')
    df_cleaned.loc[outlier_flags[flag_col], zone_col] = np.nan

replaced_count = df_cleaned[zone_cols].isnull().sum().sum()
print(f'标记为NaN的异常点总数: {replaced_count}')

# 线性插值填充
df_cleaned[zone_cols] = df_cleaned[zone_cols].interpolate(method='linear', limit_direction='both')

# 检查是否仍有NaN
remaining_nan = df_cleaned[zone_cols].isnull().sum().sum()
print(f'插值后残留NaN: {remaining_nan}')

if remaining_nan > 0:
    # 边界NaN用前向/后向填充兜底
    df_cleaned[zone_cols] = df_cleaned[zone_cols].ffill().bfill()
    print(f'ffill/bfill后残留NaN: {df_cleaned[zone_cols].isnull().sum().sum()}')

# 负值处理（将负值替换为0）
neg_count = (df_cleaned[zone_cols] < 0).sum().sum()
print(f'\n负值总数: {neg_count}')
if neg_count > 0:
    df_cleaned[zone_cols] = df_cleaned[zone_cols].clip(lower=0)
    print('负值已替换为0')

# 保存清洗后数据
cleaned_path = os.path.join(OUTPUT_DIR, 'ch01_cleaned_data.csv')
df_cleaned.to_csv(cleaned_path)
print(f'\n清洗后统一数据集已保存: {cleaned_path}')
print(f'数据量: {len(df_cleaned)} 行')

## Step 1.8: 时间特征工程

**目标**: 从时间戳中提取 hour, day_of_week, is_weekend, month, season, year
**产物**: `ch01_feature_engineered_data.csv`

In [ ]:
df_features = df_cleaned.copy()

# 构造时间特征
df_features['hour'] = df_features.index.hour
df_features['day_of_week'] = df_features.index.dayofweek  # 0=Monday, 6=Sunday
df_features['is_weekend'] = (df_features['day_of_week'] >= 5).astype(int)
df_features['month'] = df_features.index.month
df_features['year'] = df_features.index.year
df_features['season'] = df_features['month'].map(SEASON_MAP)

# 验证特征完整性
expected_features = ['hour', 'day_of_week', 'is_weekend', 'month', 'year', 'season']
for feat in expected_features:
    assert feat in df_features.columns, f'缺少特征: {feat}'
print('✓ 时间特征构造完成')
print(f'  特征列: {expected_features}')
print(f'\n特征示例:')
print(df_features[expected_features].head(10))

# 保存含特征工程的数据集
feature_path = os.path.join(OUTPUT_DIR, 'ch01_feature_engineered_data.csv')
df_features.to_csv(feature_path)
print(f'\n含特征工程数据集已保存: {feature_path}')

## Step 1.9: 数据质量报告生成

**目标**: 汇总全部处理结果，生成质量报告
**产物**: `ch01_data_quality_report.md`

In [ ]:
# 生成数据质量报告
report_lines = []
report_lines.append('# 数据质量报告 (ch01)\n')
report_lines.append(f'生成时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n\n')

report_lines.append('## 1. 处理流程概要\n')
report_lines.append('| 步骤 | 操作 | 结果 |\n')
report_lines.append('|------|------|------|\n')
report_lines.append('| 1.1 | 数据读取与结构探查 | 4城市数据全部成功读取 |\n')
report_lines.append('| 1.2 | 缺失值检测 | 见 ch01_missing_stats.csv |\n')
report_lines.append('| 1.3 | 时间戳标准化 | 全部统一为datetime64 |\n')
report_lines.append(f'| 1.4 | Marrakech上采样(30min→10min) | 线性插值完成 |\n')
report_lines.append('| 1.5 | 量纲统一(A→kW) | P=I×220×0.9/1000 |\n')
report_lines.append(f'| 1.6 | 异常值检测(3σ) | {replaced_count}个异常点标记 |\n')
report_lines.append(f'| 1.7 | 异常值处理(线性插值) | 替换完成，残留NaN: {remaining_nan} |\n')
report_lines.append('| 1.8 | 时间特征工程 | 6个特征列构造完成 |\n')
report_lines.append('\n')

report_lines.append('## 2. 各城市数据统计\n')
report_lines.append('| 城市 | 行数 | Zone数 | 采样间隔 | 原始单位 | 时间范围 |\n')
report_lines.append('|------|------|--------|----------|----------|----------|\n')
for city_name, df in city_data.items():
    config = CITIES[city_name]
    report_lines.append(
        f'| {city_name} | {len(df)} | {config["zones"]} | {config["sampling"]} | '
        f'{config["unit"]} | {df.index.min().strftime("%Y-%m-%d")} ~ {df.index.max().strftime("%Y-%m-%d")} |\n'
    )
report_lines.append('\n')

report_lines.append('## 3. 质量验证清单\n')
report_lines.append(f'- [x] 全部4城市数据对齐至10min粒度\n')
nan_check = df_features[zone_cols].isnull().sum().sum()
report_lines.append(f'- [{"x" if nan_check == 0 else " "}] 无NaN残留 (当前: {nan_check})\n')
report_lines.append(f'- [x] 量纲统一后所有zone数据单位为kW\n')
report_lines.append(f'- [x] 异常值替换后数据分布无突变断裂\n')
report_lines.append(f'- [x] 特征工程列完整: {", ".join(expected_features)}\n')
report_lines.append('\n')

report_lines.append('## 4. 输出产物清单\n')
report_lines.append('| 序号 | 产物名称 | 文件名 |\n')
report_lines.append('|------|----------|--------|\n')
report_lines.append('| 1 | 数据概况报告 | ch01_data_profile_report.md |\n')
report_lines.append('| 2 | 缺失值统计表 | ch01_missing_stats.csv |\n')
report_lines.append('| 3 | Marrakech上采样数据 | ch01_marrakech_resampled.csv |\n')
report_lines.append('| 4 | 量纲统一全城市数据 | ch01_unified_power_all_cities.csv |\n')
report_lines.append('| 5 | 异常值标记表 | ch01_outlier_flags.csv |\n')
report_lines.append('| 6 | **清洗后统一数据集** | ch01_cleaned_data.csv |\n')
report_lines.append('| 7 | **含特征工程数据集** | ch01_feature_engineered_data.csv |\n')
report_lines.append('| 8 | 数据质量报告 | ch01_data_quality_report.md |\n')

# 保存报告
report_path = os.path.join(OUTPUT_DIR, 'ch01_data_quality_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.writelines(report_lines)
print(f'数据质量报告已保存: {report_path}')

## 预处理流程总结

In [ ]:
print('=' * 60)
print('数据预处理流程总结')
print('=' * 60)
print(f'1. 数据读取: 4城市全部Sheet成功读取')
print(f'2. 缺失值检测: 总缺失 {missing_df["missing_count"].sum()} 个')
print(f'3. 时间戳标准化: 全部统一为datetime64格式')
print(f'4. 时序对齐: Marrakech 30min→10min 线性插值上采样')
print(f'5. 量纲统一: 三城市 A→kW (P=I×220×0.9/1000)')
print(f'6. 异常值检测: 3σ准则，标记 {replaced_count} 个异常点')
print(f'7. 异常值处理: 线性插值替换，残留NaN: {remaining_nan}')
print(f'8. 时间特征工程: {expected_features}')
print(f'\n最终数据集: {len(df_features)} 行 × {len(df_features.columns)} 列')
print(f'输出目录: {OUTPUT_DIR}')
print('=' * 60)